In [2]:
import pandas as pd
import numpy as np
import csv

In [7]:
data = pd.read_csv("gpu.csv")

print(data.isna().sum())

timestamp               0
 index                  0
 power.draw [W]         0
 utilization.gpu [%]    0
 temperature.gpu        0
dtype: int64


In [3]:
FILENAME = "gpu.csv"


def pureData(dataframe):
    dataframe.columns = dataframe.columns.str.strip()
    dataframe["power.draw [W]"] = (
        dataframe["power.draw [W]"].str.replace(" W", "", regex=False).astype(float)
    )
    dataframe["utilization.gpu [%]"] = (
        dataframe["utilization.gpu [%]"].str.replace(" %", "", regex=False).astype(int)
    )
    dataframe["timestamp"] = pd.to_datetime(dataframe["timestamp"])
    dataframe = dataframe.sort_values("timestamp")
    return dataframe

def getStats(filename):
    data = pureData(pd.read_csv(filename))
    power = data["power.draw [W]"]
    avg_wattage = power.mean()
    max_wattage = power.max()


    time = data["timestamp"].diff().dt.total_seconds() / 3600
    complete_wattage = (data["power.draw [W]"].shift(1) * time).sum()


    training_time = (
        data["timestamp"].iloc[-1] - data["timestamp"].iloc[0]
    ).total_seconds()

    print(f"Average wattage: {avg_wattage:.3f} W")
    print(f"Max wattage: {max_wattage:.3f} W")
    print(f"Training time: {training_time:.3f} seconds, or {(training_time / 60):.3f} minutes")
    print(f"Complete Wattage: {complete_wattage:.3f} Wh")


getStats(FILENAME)

Average wattage: 166.600 W
Max wattage: 175.880 W
Training time: 3498.986 seconds, or 58.316 minutes
Complete Wattage: 161.891 Wh


In [ ]:
#FOR THE DMON TXT FILE INSTEAD OF CSV

FILENAME = "gpu.log.webserver.txt"

def getDmonStats(filename):
    with open(filename, "r", encoding="utf-16") as f:
        lines = f.readlines()

    headers = lines[0].lstrip("#").split()

    rows = []

    for line in lines[2:]:
        values = line.split()
        if len(values) == len(headers):
            rows.append(values)

    data = pd.DataFrame(rows, columns=headers)
    data["timestamp"] = pd.to_datetime(data["Date"] + " " + data["Time"], format="%Y%m%d %H:%M:%S")
    data["pwr"] = pd.to_numeric(data["pwr"], errors="coerce")

    power = data["pwr"]
    avg_wattage = power.mean()
    max_wattage = power.max()

    time = data["timestamp"].diff().dt.total_seconds() / 3600
    complete_wattage = (data["pwr"].shift(1) * time).sum()

    starting_time = (
        data["timestamp"].iloc[-1] - data["timestamp"].iloc[0]
    ).total_seconds()

    print(f"Average wattage: {avg_wattage:.3f} W")
    print(f"Max wattage: {max_wattage:.3f} W")
    print(f"Starting time: {starting_time:.3f} seconds, or {(starting_time / 60):.3f} minutes")
    print(f"Complete Wattage: {complete_wattage:.3f} Wh")


getDmonStats(FILENAME)

Average wattage: 6.197 W
Max wattage: 36.000 W
Starting time: 274.000 seconds, or 4.567 minutes
Complete Wattage: 0.397 Kwh


: 